In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
pip install jiwer

## Imports and Seeds

In [2]:
import os, random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Data Loading

In [4]:
DATA_DIR = "/kaggle/input/datasets/bryanpark/korean-single-speaker-speech-dataset"
AUDIO_DIR = f"{DATA_DIR}/kss"

MODEL_NAME = "openai/whisper-small"
SAMPLING_RATE = 16_000   # whisper's rate; KSS is 44.1kHz

## Load the transcript file

#### I decided to use the expanded script as the training target, not the original. Reason: the original writes "20분" but the voice actually says "이십 분" — the expanded version matches the sounds. Training a model to output "20" when it hears "이십" teaches it to hallucinate.

In [5]:
cols = ["path", "original", "expanded", "decomposed", "duration", "translation"]
df = pd.read_csv(f"{DATA_DIR}/transcript.v.1.4.txt", sep="|", names=cols)

df["path"] = AUDIO_DIR + "/" + df["path"]
df = df.rename(columns={"expanded": "text"})[["path", "text", "duration"]]

print(len(df))
print(df["duration"].describe())
df.head(3)

12854
count    12854.000000
mean         3.601431
std          1.006961
min          0.800000
25%          2.900000
50%          3.500000
75%          4.200000
max          9.000000
Name: duration, dtype: float64


,path,text,duration
0,/kaggle/input/datasets/bryanpark/korean-single...,그는 괜찮은 척하려고 애쓰는 것 같았다.,3.5
1,/kaggle/input/datasets/bryanpark/korean-single...,그녀의 사랑을 얻기 위해 애썼지만 헛수고였다.,4.0
2,/kaggle/input/datasets/bryanpark/korean-single...,용돈을 아껴 써라.,1.8


## Filter & Split

#### Here, I am cutting the length of the audio for better training (Whispr's limit). But most korean words are short, so there might be no changes at all


In [6]:
from sklearn.model_selection import train_test_split

df = df[df["duration"] <= 25.0]

train_df, eval_df = train_test_split(df, test_size=0.05, random_state=SEED)
train_df = train_df.sample(n=3000, random_state=SEED).reset_index(drop=True)
eval_df = eval_df.sample(n=300, random_state=SEED).reset_index(drop=True)
print(len(train_df), len(eval_df))

3000 300


## Processor
#### feature extractor (waveform to log-mel spectrogram) + tokenizer (text to token ids)

In [7]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="korean", task="transcribe"
)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

## Datasets & preprocessing
#### from 44.1kHz to 16kHz

In [8]:
from datasets import Dataset, Audio

train_ds = Dataset.from_pandas(train_df[["path", "text"]], preserve_index=False)
eval_ds = Dataset.from_pandas(eval_df[["path", "text"]], preserve_index=False)
train_ds = train_ds.cast_column("path", Audio(sampling_rate=SAMPLING_RATE))
eval_ds = eval_ds.cast_column("path", Audio(sampling_rate=SAMPLING_RATE))

def prepare(batch):
    audio = batch["path"]
    batch["input_features"] = processor(
        audio["array"], sampling_rate=SAMPLING_RATE
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_ds = train_ds.map(prepare, remove_columns=["path", "text"])
eval_ds = eval_ds.map(prepare, remove_columns=["path", "text"])

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

## Data collator

In [10]:
class DataCollatorSpeechSeq2Seq:
    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2Seq()

## Metrics: WER & CER
#### WER - word error rate, CER - character error rate

In [12]:
from jiwer import wer, cer

def compute_metrics(pred):
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer(label_str, pred_str), "cer": cer(label_str, pred_str)}

## Model

In [13]:
from transformers import (WhisperForConditionalGeneration,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = "korean"
model.generation_config.task = "transcribe"
model.config.use_cache = False

args = Seq2SeqTrainingArguments(
    output_dir="whisper-kss",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    num_train_epochs=2,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=data_collator, compute_metrics=compute_metrics,
)

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
print(trainer.evaluate())   # zero-shot Whisper on Korean
trainer.train()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

{'eval_loss': 4.635531902313232, 'eval_wer': 0.1895551257253385, 'eval_cer': 0.06088306312521559, 'eval_runtime': 109.1248, 'eval_samples_per_second': 2.749, 'eval_steps_per_second': 0.348}


Epoch,Training Loss,Validation Loss,Wer,Cer
1,No log,0.133314,0.125081,0.040186
2,No log,0.112652,0.119278,0.038462


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=188, training_loss=3.8840841739735703, metrics={'train_runtime': 2953.62, 'train_samples_per_second': 2.031, 'train_steps_per_second': 0.064, 'total_flos': 1.73151240192e+18, 'train_loss': 3.8840841739735703, 'epoch': 2.0})

## Some results

In [15]:
sample = eval_ds.select(range(5))
preds = trainer.predict(sample)
pred_ids = preds.predictions
pred_ids[pred_ids == -100] = processor.tokenizer.pad_token_id
for p, ref in zip(processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True),
                  eval_df["text"].head(5)):
    print("PRED:", p)
    print("REF :", ref)
    print()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


PRED: 여기는 햇빛이 잘 안 들어요.
REF : 여기는 햇빛이 잘 안 들어요.

PRED: 그는 할아버지로부터 막대한 재산을 물려받았다.
REF : 그는 할아버지로부터 막대한 재산을 물려받았다.

PRED: 침으로 편지 봉투에 우표를 붙이지 마세요.
REF : 침으로 편지 봉투에 우표를 붙이지 마세요.

PRED: 자기는 나를 얼마나 사랑해?
REF : 자기는 날 얼마나 사랑해?

PRED: 놀라서 목소리가 떨리기 시작했다.
REF : 놀라서 목소리가 떨리기 시작했다.

